# Selected 6 Planets — Transit Extraction (3T Windows, All Sectors)
**UHJ:** WASP-078, WASP-072, TOI-4773  
**HJ:** TrEs-5, CoRoT-01, HATS-13

Pipeline per planet:
1. Download all TESS SPOC 2-min sectors and stitch into one time/flux array
2. Phase-fold the full dataset → empirical T0
3. Predict all nominal transit midpoints across all sectors
4. Refine each midpoint empirically (flux-weighted centroid; fallback to local min if shift > 0.5×T14; reject if SNR < 2)
5. Extract **3T window** = `[t_mid − 1.5×T14,  t_mid + 1.5×T14]` — reject if any gap > 5 min falls inside
6. Plot every valid window

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
import lightkurve as lk

warnings.filterwarnings('ignore')

MAX_GAP_MIN    = 5.0
SNR_THRESHOLD  = 2.0
MAX_SHIFT_FRAC = 0.5
N_T0_SCAN      = 1000   # resolution of T0 scan: P/1000 (~2-3 min for typical periods)

R_sun = 6.957e8; R_jup = 7.149e7; AU = 1.496e11

PLANETS = {
    'WASP-078': dict(kind='UHJ', color='#c0392b', P=2.175, depth=0.811, Rp=2.060, Rs=2.350, a=0.03670),
    'WASP-072': dict(kind='UHJ', color='#c0392b', P=2.217, depth=0.434, Rp=1.270, Rs=1.980, a=0.03708),
    'TOI-4773': dict(kind='UHJ', color='#c0392b', P=1.745, depth=0.811, Rp=1.358, Rs=1.550, a=0.03216),
    'TrEs-5':   dict(kind='HJ',  color='#1a6b3c', P=1.482, depth=1.998, Rp=1.194, Rs=0.868, a=0.02446),
    'CoRoT-01': dict(kind='HJ',  color='#1a6b3c', P=1.509, depth=1.986, Rp=1.551, Rs=1.131, a=0.02536),
    'HATS-13':  dict(kind='HJ',  color='#1a6b3c', P=3.044, depth=1.972, Rp=1.212, Rs=0.887, a=0.04057),
}

def estimate_T14(P_days, Rs, Rp, a_AU):
    return (P_days / np.pi) * np.arcsin(np.clip((Rs*R_sun + Rp*R_jup) / (a_AU*AU), 0, 1))

def clean_lc(lc, sigma=3.0):
    lc   = lc.remove_nans().remove_outliers(sigma=sigma)
    time = np.asarray(lc.time.value, dtype=float)
    flux = np.asarray(lc.flux.value, dtype=float)
    flux = flux / np.nanmedian(flux)
    return time, flux

def find_empirical_T0(time, flux, period, T14_d, n_scan=1000):
    """
    Scan T0 values from time[0] to time[0]+P and find which minimizes
    the median in-transit flux across all transits. More robust than
    phase-bin minimum for shallow or noisy transits.
    """
    half         = T14_d / 2.0
    t0_candidates = np.linspace(time[0], time[0] + period, n_scan, endpoint=False)
    best_score   = np.inf
    best_t0      = t0_candidates[0]

    for t0 in t0_candidates:
        phase   = (time - t0) % period
        in_mask = (phase <= half) | (phase >= period - half)
        if in_mask.sum() < 3:
            continue
        score = np.nanmedian(flux[in_mask])
        if score < best_score:
            best_score = score
            best_t0    = t0

    return float(best_t0)

def refine_midpoint(time, flux, t_nominal, T14_d,
                    snr_threshold=2.0, max_shift_frac=0.5):
    """
    Refine predicted midpoint to empirical flux minimum.
    Returns (t_refined, depth, snr, note)  note in {'GOOD','ADJUSTED','NO_DIP'}
    """
    half     = T14_d / 2.0
    in_mask  = (time >= t_nominal - half)       & (time <= t_nominal + half)
    out_mask = ((time >= t_nominal - 1.5*T14_d) & (time <  t_nominal - half)) | \
               ((time >  t_nominal + half)       & (time <= t_nominal + 1.5*T14_d))

    if in_mask.sum() < 3 or out_mask.sum() < 3:
        return float(t_nominal), 0.0, 0.0, 'NO_DIP'

    baseline = np.nanmedian(flux[out_mask])
    noise    = np.nanstd(flux[out_mask])
    depth    = baseline - np.nanmedian(flux[in_mask])
    snr      = depth / noise if noise > 0 else 0.0

    if snr < snr_threshold:
        return float(t_nominal), float(depth), float(snr), 'NO_DIP'

    t_in    = time[in_mask]
    f_in    = flux[in_mask]
    weights = np.clip(baseline - f_in, 0, None)
    if weights.sum() == 0:
        return float(t_nominal), float(depth), float(snr), 'NO_DIP'

    t_centroid = float(np.sum(weights * t_in) / np.sum(weights))
    shift      = abs(t_centroid - t_nominal)

    if shift > max_shift_frac * T14_d:
        scan_mask  = (time >= t_nominal - T14_d) & (time <= t_nominal + T14_d)
        if scan_mask.sum() >= 3:
            t_centroid = float(time[scan_mask][np.argmin(flux[scan_mask])])
        note = 'ADJUSTED'
    else:
        note = 'GOOD'

    return t_centroid, float(depth), float(snr), note

def window_has_gap(time, t_lo, t_hi, max_gap_min=5.0):
    mask = (time >= t_lo) & (time <= t_hi)
    t_w  = time[mask]
    if len(t_w) < 2:
        return True
    return bool(np.any(np.diff(t_w) > max_gap_min / 1440.0))

print('Setup complete.')


In [ ]:
# ── DIAGNOSTIC CELL ──────────────────────────────────────────────────────────
# Run this BEFORE the main extraction cell.
# Shows phase-fold plot (verify dip is centred) + per-midpoint SNR table.

DIAG_PLANETS = list(PLANETS.keys())   # edit to ['HATS-13'] to debug just one

for planet in DIAG_PLANETS:
    cfg        = PLANETS[planet]
    P          = cfg['P']
    depth      = cfg['depth']
    T14_d      = estimate_T14(P, cfg['Rs'], cfg['Rp'], cfg['a'])
    depth_frac = depth / 100.0
    color      = cfg['color']

    print(f'\n── {planet} ──')

    try:
        sr = lk.search_lightcurve(planet, mission='TESS', cadence=120, author='SPOC')
        if len(sr) == 0:
            print('  No data.'); continue
        lc_collection = sr.download_all(quality_bitmask='default')
    except Exception as e:
        print(f'  Download error: {e}'); continue

    all_time = []; all_flux = []
    for lc_raw in lc_collection:
        try:
            t, f = clean_lc(lc_raw)
            all_time.append(t); all_flux.append(f)
        except:
            continue
    if not all_time:
        print('  No usable data.'); continue

    all_time = np.concatenate(all_time)
    all_flux = np.concatenate(all_flux)
    sort_idx = np.argsort(all_time)
    all_time = all_time[sort_idx]
    all_flux = all_flux[sort_idx]

    print('  Finding T0 (scanning)...', flush=True)
    T0_emp = find_empirical_T0(all_time, all_flux, P, T14_d, N_T0_SCAN)
    print(f'  T0_emp = {T0_emp:.6f} BTJD  |  T14 = {T14_d*24:.2f} h')

    # ── Phase-fold plot ───────────────────────────────────────────────────────
    phase = ((all_time - T0_emp) % P) / P
    phase[phase > 0.5] -= 1.0   # centre dip at 0

    mg   = max(3*depth_frac, 0.003)
    zoom = 3 * T14_d / P

    fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))

    axes[0].plot(phase, all_flux, '.', ms=0.5, color=color, alpha=0.3)
    axes[0].axvline(0, color='black', lw=0.8, ls='--', alpha=0.7)
    axes[0].axvline(-0.5*T14_d/P, color='gray', lw=0.6, ls=':', alpha=0.6)
    axes[0].axvline( 0.5*T14_d/P, color='gray', lw=0.6, ls=':', alpha=0.6)
    axes[0].set_xlim(-0.5, 0.5)
    axes[0].set_ylim(1 - 4*depth_frac - mg, 1 + mg)
    axes[0].set_xlabel('Phase (T0-centred)', fontsize=10)
    axes[0].set_ylabel('Norm. Flux', fontsize=10)
    axes[0].set_title(f'{planet} — full phase fold', fontsize=10)
    axes[0].yaxis.set_major_formatter(plt.FormatStrFormatter('%.4f'))

    axes[1].plot(phase, all_flux, '.', ms=1.0, color=color, alpha=0.5)
    axes[1].axvline(0, color='black', lw=0.8, ls='--', alpha=0.7)
    axes[1].axvline(-0.5*T14_d/P, color='gray', lw=0.6, ls=':', alpha=0.6)
    axes[1].axvline( 0.5*T14_d/P, color='gray', lw=0.6, ls=':', alpha=0.6)
    axes[1].set_xlim(-zoom, zoom)
    axes[1].set_ylim(1 - 4*depth_frac - mg, 1 + mg)
    axes[1].set_xlabel('Phase (T0-centred)', fontsize=10)
    axes[1].set_title(f'{planet} — zoomed (±3T14)', fontsize=10)
    axes[1].yaxis.set_major_formatter(plt.FormatStrFormatter('%.4f'))

    plt.tight_layout()
    plt.show()

    # ── Per-midpoint SNR table ────────────────────────────────────────────────
    n_start = int(np.floor((all_time[0]  - T0_emp) / P))
    n_end   = int(np.ceil( (all_time[-1] - T0_emp) / P))
    mids = [T0_emp + n*P for n in range(n_start, n_end+1)
            if all_time[0] + 1.5*T14_d <= T0_emp + n*P <= all_time[-1] - 1.5*T14_d]

    print(f'  {"#":>3}  {"t_nom (BTJD)":>14}  {"SNR":>6}  {"depth":>8}  {"note":<10}  gap?')
    for i, t_nom in enumerate(mids):
        t_ref, dep, snr, note = refine_midpoint(
            all_time, all_flux, t_nom, T14_d, snr_threshold=0.0
        )
        t_lo = t_ref - 1.5*T14_d
        t_hi = t_ref + 1.5*T14_d
        gap  = window_has_gap(all_time, t_lo, t_hi, MAX_GAP_MIN)
        flag = 'GAP' if gap else 'ok'
        print(f'  {i+1:>3}  {t_nom:>14.6f}  {snr:>6.2f}  {dep:>8.5f}  {note:<10}  {flag}')


In [ ]:
RESULTS = {}

for planet, cfg in PLANETS.items():
    P          = cfg['P']
    depth      = cfg['depth']
    color      = cfg['color']
    T14_d      = estimate_T14(P, cfg['Rs'], cfg['Rp'], cfg['a'])
    T14_h      = T14_d * 24
    depth_frac = depth / 100.0

    print(f'\n{"="*70}')
    print(f'  {planet} ({cfg["kind"]}) | P={P:.3f} d | depth={depth:.3f}% | T14={T14_h:.2f} h')
    print(f'{"="*70}')

    # ── 1. Download all sectors ───────────────────────────────────────────────
    try:
        sr = lk.search_lightcurve(planet, mission='TESS', cadence=120, author='SPOC')
        if len(sr) == 0:
            print('  No SPOC 2-min data — skipped.'); continue
        lc_collection = sr.download_all(quality_bitmask='default')
    except Exception as e:
        print(f'  Download error: {e}'); continue

    print(f'  {len(lc_collection)} sector(s) downloaded.')

    # ── 2. Stitch all sectors ─────────────────────────────────────────────────
    all_time = []; all_flux = []
    for lc_raw in lc_collection:
        try:
            t, f = clean_lc(lc_raw)
            all_time.append(t); all_flux.append(f)
        except:
            continue

    if not all_time:
        print('  No usable data — skipped.'); continue

    all_time = np.concatenate(all_time)
    all_flux = np.concatenate(all_flux)
    sort_idx = np.argsort(all_time)
    all_time = all_time[sort_idx]
    all_flux = all_flux[sort_idx]

    # ── 3. Empirical T0 via direct scan ──────────────────────────────────────
    print('  Finding T0 (scanning)...', flush=True)
    T0_emp = find_empirical_T0(all_time, all_flux, P, T14_d, N_T0_SCAN)
    print(f'  Empirical T0 = {T0_emp:.6f} BTJD')

    # ── 4. Predict all midpoints ──────────────────────────────────────────────
    n_start = int(np.floor((all_time[0]  - T0_emp) / P))
    n_end   = int(np.ceil( (all_time[-1] - T0_emp) / P))
    midpoints_nominal = [
        T0_emp + n * P for n in range(n_start, n_end + 1)
        if all_time[0] + 1.5*T14_d <= T0_emp + n*P <= all_time[-1] - 1.5*T14_d
    ]
    print(f'  {len(midpoints_nominal)} nominal midpoints across all sectors.')

    # ── 5. Refine + extract gap-free 3T windows ───────────────────────────────
    windows = []
    n_no_dip = 0; n_gapped = 0; n_oob = 0

    for t_nom in midpoints_nominal:
        t_ref, dep, snr, note = refine_midpoint(
            all_time, all_flux, t_nom, T14_d, SNR_THRESHOLD, MAX_SHIFT_FRAC
        )

        if note == 'NO_DIP':
            n_no_dip += 1; continue

        t_lo = t_ref - 1.5 * T14_d
        t_hi = t_ref + 1.5 * T14_d

        if t_lo < all_time[0] or t_hi > all_time[-1]:
            n_oob += 1; continue

        if window_has_gap(all_time, t_lo, t_hi, MAX_GAP_MIN):
            n_gapped += 1; continue

        mask = (all_time >= t_lo) & (all_time <= t_hi)
        windows.append(dict(
            t_nom=t_nom, t_ref=t_ref, depth=dep, snr=snr, note=note,
            time=all_time[mask], flux=all_flux[mask]
        ))

    RESULTS[planet] = dict(T14_d=T14_d, windows=windows)
    print(f'  Rejected — no dip: {n_no_dip} | gapped: {n_gapped} | out of bounds: {n_oob}')
    print(f'  Valid gap-free 3T windows: {len(windows)}')

    if not windows:
        print('  Nothing to plot.'); continue

    # ── 6. Plot ───────────────────────────────────────────────────────────────
    NCOLS = 4
    nrows = int(np.ceil(len(windows) / NCOLS))
    fig, axes = plt.subplots(nrows, NCOLS,
                             figsize=(NCOLS * 4.2, nrows * 3.4),
                             squeeze=False)
    fig.suptitle(
        f'{planet} ({cfg["kind"]})  |  P={P:.3f} d  |  '
        f'depth={depth:.3f}%  |  T14={T14_h:.2f} h  |  {len(windows)} valid transits',
        fontsize=12, fontweight='bold', y=1.01
    )

    half_T14_min = (T14_d / 2) * 1440

    for idx, w in enumerate(windows):
        ri = idx // NCOLS
        ci = idx  % NCOLS
        ax = axes[ri][ci]

        x = (w['time'] - w['t_ref']) * 1440.0
        f = w['flux']

        ax.plot(x, f, '.', ms=2.5, color=color, alpha=0.8, zorder=3)
        ax.axvline(0,             color='black', lw=0.8, ls='--', alpha=0.7, zorder=4)
        ax.axvline(-half_T14_min, color='gray',  lw=0.6, ls=':',  alpha=0.5)
        ax.axvline( half_T14_min, color='gray',  lw=0.6, ls=':',  alpha=0.5)

        fp1  = np.nanpercentile(f, 1)
        fp99 = np.nanpercentile(f, 99)
        mg   = max(3*depth_frac, (fp99-fp1)*0.1, 0.002)
        ax.set_ylim(min(fp1, 1.0 - 2*depth_frac) - mg*0.3,
                    max(fp99, 1.0 + depth_frac)   + mg*0.3)

        ax.set_title(f'T{idx+1}  {w["note"]}  SNR={w["snr"]:.1f}', fontsize=8)
        ax.set_xlabel('min from midpoint', fontsize=7)
        ax.tick_params(labelsize=7)
        ax.yaxis.set_major_formatter(plt.FormatStrFormatter('%.4f'))

    for idx in range(len(windows), nrows * NCOLS):
        axes[idx // NCOLS][idx % NCOLS].set_visible(False)

    plt.tight_layout()
    plt.show()
    print(f'  Done.\n', flush=True)

print('\nAll planets processed.')
print('\nSummary:')
for planet, res in RESULTS.items():
    print(f'  {planet}: {len(res["windows"])} valid transits')
